<a href="https://colab.research.google.com/github/Vivekb638/movie-recommendation-pyspark/blob/main/notebooks/04_als_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install and initialize PySpark

In [2]:
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark==3.5.5

Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.018s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.018s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
done.
Setting up openjdk-17-jdk-h

In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MovieRecommendationALS")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


Load the cleaned ratings

In [4]:
PROJECT_DIR = "/content/drive/MyDrive/movie-recommendation-pyspark"

PROCESSED_DIR = os.path.join(
    PROJECT_DIR, "data", "processed"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR, "models"
)

RESULTS_DIR = os.path.join(
    PROJECT_DIR, "results"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

ratings_clean = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "ratings_clean")
)

movies_clean = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "movies_clean")
)

ratings_clean.show(5)
print("Total ratings:", ratings_clean.count())

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|   3408|   4.0|978300275|
|     1|   2791|   4.0|978302188|
|     1|   2018|   4.0|978301777|
|     1|   2321|   3.0|978302205|
|     1|   2340|   3.0|978300103|
+------+-------+------+---------+
only showing top 5 rows

Total ratings: 1000209


Prepare the ML dataset

In [5]:
from pyspark.sql.functions import col

als_data = ratings_clean.select(
    col("userId").cast("int"),
    col("movieId").cast("int"),
    col("rating").cast("float")
).dropna()

als_data.printSchema()
als_data.show(5)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   3408|   4.0|
|     1|   2791|   4.0|
|     1|   2018|   4.0|
|     1|   2321|   3.0|
|     1|   2340|   3.0|
+------+-------+------+
only showing top 5 rows



Split the data into training and testing datasets

In [6]:
train_data, test_data = als_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

Training records: 800056
Testing records: 200153


Train the ALS recommendation model

In [7]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

als_model = als.fit(train_data)

print("ALS model training completed!")

ALS model training completed!


Generate predictions

In [8]:
predictions = als_model.transform(test_data)

predictions.show(10)

print("Prediction records:", predictions.count())

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|    12|    919|   5.0| 3.4962685|
|    13|    110|   4.0| 3.8730578|
|    13|    165|   3.0|  3.080477|
|    13|    296|   4.0|  3.448441|
|    13|   1544|   4.0|  2.889644|
|    13|   1580|   4.0| 3.3703747|
|    13|   2414|   2.0|  3.089882|
|    13|   2686|   4.0|   3.41571|
|    13|   2968|   3.0|  3.050103|
|    13|   3036|   3.0| 2.6754916|
+------+-------+------+----------+
only showing top 10 rows

Prediction records: 200137


Evaluate the model using RMSE

In [9]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator_rmse.evaluate(predictions)

print("ALS Model RMSE:", rmse)

ALS Model RMSE: 0.8704397715424746


Evaluate using MAE

In [10]:
evaluator_mae = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

mae = evaluator_mae.evaluate(predictions)

print("ALS Model MAE:", mae)

ALS Model MAE: 0.6981796986320095


In [11]:
import json

metrics = {
    "algorithm": "ALS",
    "rank": 20,
    "maxIter": 10,
    "regParam": 0.1,
    "rmse": float(rmse),
    "mae": float(mae)
}

metrics_path = os.path.join(
    RESULTS_DIR, "als_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("Evaluation metrics saved:", metrics_path)

Evaluation metrics saved: /content/drive/MyDrive/movie-recommendation-pyspark/results/als_metrics.json


Generate movie recommendations

In [12]:
user_recommendations = als_model.recommendForAllUsers(10)

user_recommendations.show(5, truncate=False)

+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                         |
+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|5     |[{557, 4.404918}, {1420, 4.1763163}, {1423, 4.173568}, {1138, 4.0836263}, {1192, 4.012525}, {2503, 3.9496086}, {2997, 3.9285228}, {1169, 3.9248445}, {106, 3.9245586}, {1199, 3.919936}]|
|6     |[{572, 5.4701924}, {687, 4.9349117}, {2197, 4.871375}, {3853, 4.703036}, {37, 4.655585}, {1035, 4.644765}, {1117, 4.6362824}, {985, 4.608682}, {3233, 4.571062}, {666, 4.552206}]       |
|9     |[{572, 4.8764644}, {31

In [14]:
sample_user_id = 1

single_user = spark.createDataFrame(
    [(sample_user_id,)],
    ["userId"]
)

recommendations = als_model.recommendForUserSubset(
    single_user,
    10
)

recommendations.show(truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                 |
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1     |[{572, 5.4207582}, {3233, 4.596542}, {527, 4.591734}, {318, 4.5668745}, {3517, 4.5141053}, {953, 4.47643}, {858, 4.461528}, {128, 4.451925}, {593, 4.421972}, {1207, 4.4201097}]|
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+

+------+-------------------------------------------------------------

Display movie titles with recommendations

In [15]:
from pyspark.sql.functions import explode

user_recommendations_flat = (
    user_recommendations
    .select(
        "userId",
        explode("recommendations").alias("recommendation")
    )
    .select(
        "userId",
        col("recommendation.movieId").alias("movieId"),
        col("recommendation.rating").alias("predicted_rating")
    )
)

final_recommendations = (
    user_recommendations_flat
    .join(
        movies_clean.select("movieId", "title", "genres"),
        on="movieId",
        how="inner"
    )
)

final_recommendations.filter(
    col("userId") == sample_user_id
).orderBy(
    col("predicted_rating").desc()
).show(10, truncate=False)

+-------+------+----------------+--------------------------------+------------------+
|movieId|userId|predicted_rating|title                           |genres            |
+-------+------+----------------+--------------------------------+------------------+
|572    |1     |5.4207582       |Foreign Student (1994)          |Drama             |
|3233   |1     |4.596542        |Smashing Time (1967)            |Comedy            |
|527    |1     |4.591734        |Schindler's List (1993)         |Drama|War         |
|318    |1     |4.5668745       |Shawshank Redemption, The (1994)|Drama             |
|3517   |1     |4.5141053       |Bells, The (1926)               |Crime|Drama       |
|953    |1     |4.47643         |It's a Wonderful Life (1946)    |Drama             |
|858    |1     |4.461528        |Godfather, The (1972)           |Action|Crime|Drama|
|128    |1     |4.451925        |Jupiter's Wife (1994)           |Documentary       |
|593    |1     |4.421972        |Silence of the Lambs,

Save the trained model

In [16]:
ALS_MODEL_PATH = os.path.join(
    MODEL_DIR, "als_model"
)

als_model.write().overwrite().save(ALS_MODEL_PATH)

print("ALS model saved:", ALS_MODEL_PATH)

ALS model saved: /content/drive/MyDrive/movie-recommendation-pyspark/models/als_model


In [18]:
# Save user latent factors
als_model.userFactors.write.mode("overwrite").parquet(
    os.path.join(MODEL_DIR, "user_factors")
)

# Save movie latent factors
als_model.itemFactors.write.mode("overwrite").parquet(
    os.path.join(MODEL_DIR, "item_factors")
)

print("User and item factors saved successfully!")

User and item factors saved successfully!
